# 🛠️ Chapter 03-01b: Hands-On — Your First LangChain Chains

---

## Time to Build Real Chains! 🔗

Theory is great, but chains are meant to be BUILT. In this notebook, you'll:
- ✅ Create Chat Models and test different configurations
- ✅ Master message types (System, Human, AI)
- ✅ Build chains with ChatPromptTemplate + LCEL
- ✅ Use StrOutputParser for clean text output
- ✅ Build a real translation chain

Let's chain things together! 🚀

---

## 📦 Setup

In [ ]:
# !pip install langchain-core langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

print("✅ Ready!" if os.getenv("OPENAI_API_KEY") else "❌ Set OPENAI_API_KEY in .env")

---

## 🧪 Exercise 1: Chat Models — Creating and Configuring

In [ ]:
# Create models with different configs

# Default: gpt-4o-mini, temp=0.7
llm_default = ChatOpenAI(model="gpt-4o-mini")

# Precise: temp=0 for deterministic output
llm_precise = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Creative: temp=1.0, higher max tokens
llm_creative = ChatOpenAI(model="gpt-4o-mini", temperature=1.0, max_tokens=500)

# Test the same prompt with different temperatures
prompt = "Write a one-sentence tagline for a coffee shop called 'Byte & Brew'."

print("🎯 PRECISE (temp=0):")
for i in range(3):
    result = llm_precise.invoke(prompt)
    print(f"   Run {i+1}: {result.content}")

print("\n🎨 CREATIVE (temp=1.0):")
for i in range(3):
    result = llm_creative.invoke(prompt)
    print(f"   Run {i+1}: {result.content}")

print("\n💡 Notice: temp=0 gives identical results. temp=1.0 varies each time.")

---

## 🧪 Exercise 2: Message Types in Action

In [ ]:
# Using explicit message objects
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Method 1: Message objects
messages = [
    SystemMessage(content="You are a pirate. Respond in pirate speak. Keep it under 2 sentences."),
    HumanMessage(content="What is machine learning?")
]

result = llm.invoke(messages)
print("🏴‍☠️ Pirate ML explanation:")
print(f"   {result.content}")
print(f"   Type: {type(result).__name__}")
print(f"   Token usage: {result.usage_metadata}")

In [ ]:
# Method 2: Tuple shorthand (cleaner!)
messages = [
    ("system", "You are a Shakespearean actor. Respond in iambic pentameter. 2 lines max."),
    ("human", "What is Python?")
]

result = llm.invoke(messages)
print("🎭 Shakespeare's Python:")
print(f"   {result.content}")

In [ ]:
# Multi-turn conversation (include AI messages for context)
messages = [
    ("system", "You are a helpful coding tutor. Be concise."),
    ("human", "What is a list in Python?"),
    ("ai", "A list is an ordered, mutable collection: my_list = [1, 2, 3]"),
    ("human", "How do I add an item to it?")  # The model has context!
]

result = llm.invoke(messages)
print("💬 Multi-turn conversation:")
print(f"   {result.content}")
print("\n💡 The model remembers the previous exchange because we passed it!")

---

## 🧪 Exercise 3: ChatPromptTemplate — Reusable Prompts

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create a reusable template
explain_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {domain} teacher. Explain concepts simply in 2-3 sentences."),
    ("human", "Explain {concept} to me like I'm a {level} student.")
])

# Inspect the template
print("📋 Template variables:", explain_prompt.input_variables)

# Format with different inputs
messages = explain_prompt.format_messages(
    domain="computer science",
    concept="recursion",
    level="beginner"
)

print("\n📝 Formatted messages:")
for msg in messages:
    print(f"   [{type(msg).__name__}]: {msg.content}")

# Send to model
result = llm.invoke(messages)
print(f"\n🤖 Response: {result.content}")

In [ ]:
# Reuse the SAME template with different inputs
tests = [
    {"domain": "physics", "concept": "quantum entanglement", "level": "high school"},
    {"domain": "finance", "concept": "compound interest", "level": "beginner"},
    {"domain": "biology", "concept": "CRISPR gene editing", "level": "advanced"},
]

for test in tests:
    result = llm.invoke(explain_prompt.format_messages(**test))
    print(f"📚 {test['domain'].title()} — {test['concept']}:")
    print(f"   {result.content}\n")

---

## 🧪 Exercise 4: Building Chains with LCEL

In [ ]:
# The LCEL way: prompt | model

# Step 1: Prompt template
joke_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a comedian. Tell short, family-friendly jokes."),
    ("human", "Tell me a joke about {topic}")
])

# Step 2: Chain = prompt | model
joke_chain = joke_prompt | llm

# Step 3: Invoke the chain
result = joke_chain.invoke({"topic": "programming"})
print(f"😂 Joke: {result.content}")
print(f"   Type: {type(result).__name__}")  # AIMessage

In [ ]:
# Add StrOutputParser to get just the text (not AIMessage)

from langchain_core.output_parsers import StrOutputParser

# prompt | model | parser
joke_chain_v2 = joke_prompt | llm | StrOutputParser()

result = joke_chain_v2.invoke({"topic": "data science"})
print(f"😂 Joke: {result}")
print(f"   Type: {type(result).__name__}")  # str (not AIMessage!)

print("\n💡 StrOutputParser extracts .content from the AIMessage → clean string")

---

## 🚀 Build: Translation Chain

Let's build a full translation pipeline using everything we've learned.

In [ ]:
# Full translation chain

# Step 1: Template
translate_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a professional translator.
Translate the text naturally, preserving tone and meaning.
Return ONLY the translation, nothing else."""),
    ("human", "Translate this {source} text to {target}:\n\n{text}")
])

# Step 2: Chain
translate_chain = translate_prompt | llm | StrOutputParser()

# Step 3: Test with various languages
translations = [
    {"source": "English", "target": "Hindi", "text": "Artificial intelligence is transforming every industry."},
    {"source": "English", "target": "Spanish", "text": "The quick brown fox jumps over the lazy dog."},
    {"source": "English", "target": "Japanese", "text": "I love building AI applications with Python."},
    {"source": "English", "target": "French", "text": "Machine learning models learn patterns from data."},
]

print("🌐 TRANSLATION CHAIN RESULTS")
print("=" * 60)

for t in translations:
    result = translate_chain.invoke(t)
    print(f"\n🔤 {t['source']} → {t['target']}")
    print(f"   Input:  {t['text']}")
    print(f"   Output: {result}")

In [ ]:
# Bonus: Chain with back-translation (translate and translate back)
# This is a real-world technique to verify translation quality!

original = "Debugging code at 3 AM is a rite of passage for every programmer."
language = "Mandarin Chinese"

# Forward translation
translated = translate_chain.invoke({
    "source": "English", "target": language, "text": original
})

# Back translation
back_translated = translate_chain.invoke({
    "source": language, "target": "English", "text": translated
})

print("🔄 BACK-TRANSLATION TEST")
print(f"   Original:       {original}")
print(f"   → {language}:  {translated}")
print(f"   → Back:         {back_translated}")
print(f"\n   💡 If back-translation ≈ original, the translation is likely accurate!")

---

## ✍️ Practice: Build an Email Rewriter Chain

**Challenge:** Create a chain that takes a casual email and rewrites it professionally.

In [ ]:
# YOUR CODE HERE

# TODO: Create a ChatPromptTemplate for email rewriting
#   System: "You are a professional email editor..."
#   Human: takes {tone} and {email} variables

# TODO: Build chain: prompt | llm | StrOutputParser()

# TODO: Test with:
test_email = """Hey boss, 
gonna be late tmrw cuz my car broke down. 
can I wfh? also need to talk about that raise we discussed. 
thx"""

# result = email_chain.invoke({"tone": "professional", "email": test_email})
# print(result)

<details>
<summary>💡 Click to see solution</summary>

```python
email_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a professional email editor. 
Rewrite the email in a {tone} tone. Keep the same meaning but improve grammar, 
formatting, and professionalism. Return only the rewritten email."""),
    ("human", "Rewrite this email:\n\n{email}")
])

email_chain = email_prompt | llm | StrOutputParser()

result = email_chain.invoke({"tone": "professional", "email": test_email})
print(result)
```

</details>

---

## 💡 Interview Prep: Common Questions

```
╔═══════════════════════════════════════════════════════════════════════╗
║           INTERVIEW QUESTIONS FOR THIS MODULE                        ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  Q: Build a LangChain chain from scratch. Walk me through it.       ║
║  A: 1) Create ChatPromptTemplate with system + human messages       ║
║     2) Initialize ChatOpenAI (or any model)                         ║
║     3) Chain with LCEL: chain = prompt | model | StrOutputParser()  ║
║     4) Invoke: result = chain.invoke({"var": "value"})             ║
║     The data flows: dict → template → messages → model → string    ║
║                                                                       ║
║  Q: What is StrOutputParser?                                         ║
║  A: It extracts .content from AIMessage, giving you a plain string. ║
║     Without it, you get an AIMessage object. With it, just text.    ║
║     Essential for chains where the next step expects string input.  ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 ChatOpenAI(model, temperature) creates a configured model.    │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Messages: SystemMessage, HumanMessage, AIMessage — or tuples. │
├─────────────────────────────────────────────────────────────────────┤
│  📌 ChatPromptTemplate.from_messages() for reusable prompts.      │
├─────────────────────────────────────────────────────────────────────┤
│  📌 LCEL chain: prompt | model | parser — the core pattern.       │
├─────────────────────────────────────────────────────────────────────┤
│  📌 StrOutputParser() extracts clean text from AIMessage.         │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Same chain works with different inputs — that's the power!    │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: LCEL & Runnables!

RunnablePassthrough, RunnableLambda, RunnableParallel — advanced composition, streaming, and batch processing!

---

### 🎉 You Built Your First LangChain Chains!
Templates, models, parsers, LCEL — you're building like a LangChain developer now! 🚀